# Knowledge graph entity alignment

**Difficulty:** Beginner

Decide whether two records from different sources describe the same real-world thing, with no threshold to fit: one `Score` with exactly three levels — different / related-but-unsure / same — where each level *is* the outcome (unlink, queue for a curator, merge), plus `Noul` questions riding along to tell the curator which field disagrees.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/cookbooks/entity_alignment (click to expand)</summary>

# Knowledge graph entity alignment

> Decides which of 450 candidate pairs from two beer catalogues describe the same product. One TypeSafe Score question carries the whole decision, because its three levels are the three things you can do with a pair: merge it, leave it unlinked, or hand it to a curator. There is no threshold to fit, and three Noul questions ride along in the same request to tell the curator which field the two sources disagree on.

Two data sources describe overlapping entities, and a cheap first pass has already picked out candidate pairs worth a closer look. Merging two entities inappropriately is the expensive mistake (undoing it means working out which fact came from where); missing a match only leaves a duplicate. So the judgment call needs a third option: pairs that are neither safe to merge nor safe to drop.

## Why Score, not Choice or Noul

The judgment is a `Score` with one level per outcome:

- **different product** — leave the two entities unlinked
- **related, but possibly not the same** — hand it to a curator to decide
- **same product** — merge them

`Score` attaches a semantic label directly to each outcome, including the middle one, and preserves the *ordering* between them. A `Noul` could only get there indirectly, by thresholding its single yes/no probability. A `Choice` would lose the ordered relationship entirely — "related" genuinely sits *between* "different" and "same," not off to the side.

Field-level `Noul` questions (same name? same brewery? same style?) ride along in the same request, to tell a curator *which* field the two sources disagree on when a pair lands in the middle.

## Worked examples, from the real benchmark

Real pairs from the Beer dataset (Magellan collection), text left exactly as published:

```
c446  score 1.94  confidence 0.92  ->  assert sameAs
    Thomas Hooker Old Marley Barleywine           Thomas Hooker Brewing Company   American Barleywine
    Thomas Hooker Old Marley Barleywine           Thomas Hooker Brewing Company   Barley Wine
    name 0.97   brewery 0.99   style 0.81

c428  score 1.10  confidence 0.77  ->  curator queue
    Ambleside Amber Ale                           Bridge Brewing Company          American Amber / Red A
    Bridge Ambleside Amber Ale - Pomegranate & G  Bridge Brewing Company          Amber Ale
    name 0.63   brewery 0.98   style 0.74
```

`c446` is the same beer under a slightly different style label — high score, high confidence, merge. `c428` is a beer and a fruit-and-hop *variant* of the same beer: same brewery, name only 0.63 similar, style genuinely different phrasing — the score lands in the curator range because it plausibly is, and plausibly isn't, "the same product."

## Routing: round to the nearest level, nothing to tune

```
def route(score_value):
    return OUTCOME[round(score_value)]  # nearest level names the outcome
```

There is no threshold constant anywhere — the three level descriptions *are* the whole decision, and you can write them before seeing a single score. Over all 450 candidate pairs: **40 merged (8.9%)**, **50 queued for a curator (11.1%)**, **360 left unlinked (80.0%)**. The cut points (0.5 and 1.5) aren't equally crowded — 47 pairs sit within 0.1 of the lower cut, only 9 within 0.1 of the upper one. Neither number is tuned; both follow purely from how the level descriptions are worded, and the *middle* level's wording is what moves pairs between "curator" and "unlinked."

> The full cookbook runs all 450 pairs concurrently and plots a histogram of every score against the two cut points — trimmed here to two representative pairs. Read the full cookbook at [docs.typesafe.ai/cookbooks/entity_alignment](https://docs.typesafe.ai/cookbooks/entity_alignment) any time.

</details>

## The core idea

This is the first time a `Score`'s levels don't describe *degrees of the same thing* (like severity or frustration) — they describe three genuinely different *actions* your code will take. That's a legitimate use of the primitive: as long as the outcomes have a real order (different → maybe → same is a real spectrum, not three unrelated buckets), `Score` gets you a routing decision with zero thresholds to invent, because the boundary between "unlink" and "curator" is just "closer to level 0 or level 1" — a fact about the wording, not a number you fit to data.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { score, noul } from "npm:@typesafe-ai/sdk";

const LEVELS: readonly [string, string, string] = [
  "They describe two different products.",
  "They describe closely related products that may or may not be the same one: a variant, a special edition, or a name that could plausibly refer to either.",
  "They describe one and the same product.",
];
const OUTCOME = ["leave unlinked", "curator queue", "assert sameAs"];

function route(scoreValue: number): string {
  return OUTCOME[Math.min(Math.round(scoreValue), LEVELS.length - 1)];
}

type Entity = Record<string, string>;
interface Pair {
  entityA: Entity;
  entityB: Entity;
}

// Two real pairs from the cookbook's own benchmark (Beer data, Magellan collection).
const pairs: Record<string, Pair> = {
  c446: {
    entityA: {
      name: "Thomas Hooker Old Marley Barleywine",
      brewery: "Thomas Hooker Brewing Company",
      style: "American Barleywine",
    },
    entityB: {
      name: "Thomas Hooker Old Marley Barleywine",
      brewery: "Thomas Hooker Brewing Company",
      style: "Barley Wine",
    },
  },
  c428: {
    entityA: {
      name: "Ambleside Amber Ale",
      brewery: "Bridge Brewing Company",
      style: "American Amber / Red Ale",
    },
    entityB: {
      name: "Bridge Ambleside Amber Ale - Pomegranate & Ginger",
      brewery: "Bridge Brewing Company",
      style: "Amber Ale",
    },
  },
};

async function classifyPair(id: string, pair: Pair) {
  const response = await client.systemOne({
    state: { entityA: pair.entityA, entityB: pair.entityB },
    questions: {
      linkState: score("How do the two entity descriptions relate as products?", LEVELS),
      sameName: noul("Do the two entities state the same beer name?"),
      sameBrewery: noul("Are the two entities from the same brewery?"),
      sameStyle: noul("Do the two entities describe the same beer style?"),
    },
  });

  const { linkState, sameName, sameBrewery, sameStyle } = response.answers;
  console.log(
    `${id}  score ${linkState.score.toFixed(2)}  confidence ${linkState.confidence.toFixed(2)}  -> ${
      route(linkState.score)
    }`,
  );
  console.log(
    `  name ${sameName.noul.toFixed(2)}  brewery ${sameBrewery.noul.toFixed(2)}  style ${
      sameStyle.noul.toFixed(2)
    }`,
  );
}

for (const [id, pair] of Object.entries(pairs)) {
  await classifyPair(id, pair);
}

## What just happened

Two real candidate pairs, each scored on the same three-level scale plus three field-level checks. `c446` should land near `2.0` (merge) with all three field nouls high — it's the same beer, differently worded style. `c428` should land in the curator range (roughly `1.0`–`1.5`): same brewery, similar-but-not-identical name, genuinely different style phrasing for what's arguably a variant of the same base beer.

**Try it yourself:** write a third pair that's obviously two *unrelated* beers (different name, different brewery, different style entirely) and predict where it lands — should be well below `0.5`, clearly `leave unlinked`, with all three field nouls low.

## Related

- `lessons/core-concepts/0003-score.ipynb` — the `Score` primitive, here used for routing rather than measuring a degree
- `lessons/patterns/0007-confidence-gated-routing.ipynb` — a similar three-way split (act / verify / escalate), built from thresholds on `confidence` instead of level descriptions on `score`
- `cookbooks/classification/classification-using-confidence.ipynb` — another routing decision that reads confidence to decide how specific an answer to commit to

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/typesafe-ai.md`.